<a href="https://www.kaggle.com/code/concacmemay/hybrid-rl-alns-for-vrptw?scriptVersionId=310839481" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Hybrid DDQN-ALNS for VRPTW — v9.5
## Optimizing Adaptive Large Neighbourhood Search via Distributional RL

---

### CHANGELOG v9.5 (vs v8.2):
   **[FIX-1]**  MODEL_DIR removed → all paths use CFG.output_dir
   
   **[FIX-2]**  RLALNSSolver alias → PlateauHybridSolver (fixes NameError in cells 17/20/export)
   
   **[FIX-3]**  clone_weights() added to PlateauHybridSolver
   
   **[FIX-4]**  run_benchmark accepts transfer_weights kwarg + applies before solve
   
   **[FIX-5]**  OUTPUT_DIR alias defined for export cell compatibility
   
   **[FIX-6]**  plot_transfer_comparison() defined (was referenced but missing)
   
   **[FIX-7]**  DQNNet / DQN config attrs guarded (ablation only, won't crash if skipped)
   
   **[PERF-1]** NV-increase penalty in _segment_reward (-15 per extra vehicle)
   
---

### Stops PlateauHybridSolver from trading NV for Gap%
            
   **[PERF-2]** MODE_DIVERSIFY: temp_boost lowered 1.20→1.08 to reduce NV drift
   
   **[PERF-3]** post_improve_intensify_segments: 2→3 (lock intensify longer after find)
   
---
   
 

In [1]:
# ── Cell 1 : Install, Imports & Config ─────────────────────────────────────
# !pip install numba safetensors scipy -q
 
import glob, math, os, random, time, json, shutil
from collections import deque
from dataclasses import dataclass
from typing import Deque, Dict, Iterable, List, Optional, Tuple
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from numba import njit
from scipy import stats
 
try:
    from safetensors.torch import save_file, load_file
    SAFETENSORS_OK = True
except ImportError:
    SAFETENSORS_OK = False
    print("⚠️  safetensors not available — transfer learning save/load disabled")
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'✅ Device : {DEVICE}')
 
BKS: Dict[str, Dict[str, float]] = {
    "RC101": {"nv": 14, "td": 1696.94}, "RC102": {"nv": 12, "td": 1554.75},
    "RC103": {"nv": 11, "td": 1261.67}, "RC104": {"nv": 10, "td": 1135.48},
    "RC105": {"nv": 13, "td": 1629.44}, "RC106": {"nv": 11, "td": 1424.73},
    "RC107": {"nv": 11, "td": 1230.48}, "RC108": {"nv": 10, "td": 1139.82},
    "RC201": {"nv": 4,  "td": 1406.94}, "RC202": {"nv": 3,  "td": 1365.64},
    "RC203": {"nv": 3,  "td": 1049.62}, "RC204": {"nv": 3,  "td": 798.46},
    "RC205": {"nv": 4,  "td": 1297.65}, "RC206": {"nv": 3,  "td": 1146.32},
    "RC207": {"nv": 3,  "td": 1061.14}, "RC208": {"nv": 3,  "td": 828.14},
}
 
def default_data_path() -> str:
    candidates = [
        "/kaggle/input/vrptw-benchmark-datasets/data/Solomon",
        "/kaggle/input/datasets/senju14/vrptw-benchmark-datasets/data/Solomon",
        "/content/vrptw-benchmark/data/Solomon",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return candidates[0]
 
def default_output_dir() -> str:
    return "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
 
@dataclass
class Config:
    data_path:   str   = default_data_path()
    output_dir:  str   = default_output_dir()
 
    # ALNS baseline
    alns_iterations:   int   = 2000
    destroy_ratio_min: float = 0.10
    destroy_ratio_max: float = 0.40
    temp_control:      float = 0.05
    temp_decay:        float = 0.99975
    sigma1:            int   = 33
    sigma2:            int   = 9
    sigma3:            int   = 3
    weight_decay:      float = 0.10
    segment_size:      int   = 50
    early_stop_patience: int = 400
 
    # Hybrid (PLATEAU-HYBRID / DDQN-ALNS)
    hybrid_iterations: int   = 2000
 
    # Experiment
    n_runs: int = 5
    seed:   int = 42
 
    # PlateauController network
    ctrl_state_dim:   int   = 10
    ctrl_hidden:      int   = 96
    ctrl_lr:          float = 3e-4
    ctrl_gamma:       float = 0.95
    ctrl_buffer:      int   = 4096
    ctrl_batch:       int   = 64
    ctrl_target_freq: int   = 20
    ctrl_eps_start:   float = 0.35
    ctrl_eps_end:     float = 0.05
    ctrl_eps_decay:   float = 0.992
 
    # Plateau detection & post-improve lock
    plateau_start:                   int = 60
    post_improve_intensify_segments: int = 3   # [PERF-3] was 2
 
    # [PERF-1] NV penalty coefficient in segment reward
    nv_increase_penalty: float = 15.0
 
    # DQN ablation (only used by DQNSolver)
    dqn_state_dim:      int   = 13
    dqn_hidden:         int   = 128
    dqn_lr:             float = 1e-3
    dqn_gamma:          float = 0.99
    dqn_buffer:         int   = 8192
    dqn_batch:          int   = 64
    dqn_eps_start:      float = 1.0
    dqn_eps_end:        float = 0.05
    dqn_eps_decay:      float = 0.995
    dqn_target_freq:    int   = 20
    dqn_train_freq:     int   = 5
    dqn_vehicle_penalty:float = 5.0
 
@dataclass(frozen=True)
class ModeSpec:
    name:             str
    destroy_scale:    float
    temp_boost:       float
    temp_decay_scale: float
    destroy_bias:     Tuple[float, ...]
    repair_bias:      Tuple[float, ...]
 
MODES: Tuple[ModeSpec, ...] = (
    ModeSpec("default",    1.00, 1.00,  1.000,
             (1.0, 1.0, 1.0, 1.0, 1.0), (1.0, 1.0, 1.0, 1.0)),
    ModeSpec("intensify",  0.70, 0.98,  0.995,
             (0.8, 1.3, 1.2, 0.5, 1.0), (1.3, 1.2, 0.8, 1.0)),
    # [PERF-2] temp_boost lowered 1.20 → 1.08 to reduce NV drift
    ModeSpec("diversify",  1.35, 1.08,  1.002,
             (1.3, 0.9, 1.3, 1.4, 1.0), (0.9, 1.0, 1.3, 1.0)),
    ModeSpec("tw_rescue",  1.10, 1.05,  1.000,
             (0.7, 0.9, 1.1, 0.8, 1.8), (0.8, 1.0, 1.2, 1.8)),
)
 
MODE_DEFAULT, MODE_INTENSIFY, MODE_DIVERSIFY, MODE_TW_RESCUE = 0, 1, 2, 3
CFG = Config()
 
# [FIX-5] alias for export cell
OUTPUT_DIR = CFG.output_dir
 
print('✅ Config ready.')

✅ Device : cuda
✅ Config ready.


In [2]:
# ── Cell 3 & 4: Data & Solution Model ─────────────────────────────────────────
class Inst:
    def __init__(self, raw: Dict):
        self.name     = raw["name"]
        data          = raw["data"]
        self.capacity = raw["capacity"]
        self.coords        = data[:, 1:3]
        self.demands       = data[:, 3]
        self.ready_times   = data[:, 4]
        self.due_times     = data[:, 5]
        self.service_times = data[:, 6]
        self.horizon       = self.due_times[0]
        self.n             = len(data) - 1
        diff           = self.coords[:, None, :] - self.coords[None, :, :]
        self.dist      = np.sqrt((diff ** 2).sum(axis=2))
        self.max_dist  = self.dist.max()
        self.tw_width  = self.due_times - self.ready_times
        self.max_tw_width   = self.tw_width[1:].max() + 1e-9
        self.tw_tight_frac  = sum(
            1 for i in range(1, self.n + 1)
            if self.tw_width[i] < 0.2 * self.horizon
        ) / self.n
 
 
def load_datasets(base_path: str) -> Dict[str, List[Inst]]:
    datasets: Dict[str, List[Inst]] = {}
    for group in ("rc1", "rc2"):
        files  = sorted(glob.glob(os.path.join(base_path, f"{group}*.txt")))
        insts: List[Inst] = []
        for path in files:
            with open(path) as handle:
                lines = handle.readlines()
            name     = lines[0].strip()
            capacity = float(lines[4].strip().split()[1])
            rows     = [list(map(float, l.split())) for l in lines[9:] if l.strip()]
            insts.append(Inst({"name": name, "capacity": capacity,
                               "data": np.array(rows)}))
        datasets[group] = insts
    return datasets
 
 
@njit(cache=True)
def _route_cost(route: np.ndarray, dist: np.ndarray) -> float:
    cost = dist[0, route[0]]
    for i in range(len(route) - 1):
        cost += dist[route[i], route[i + 1]]
    return cost + dist[route[-1], 0]
 
 
@njit(cache=True)
def _route_ok(route: np.ndarray, demands: np.ndarray, capacity: float,
              ready: np.ndarray, due: np.ndarray, service: np.ndarray,
              dist: np.ndarray) -> bool:
    load = 0.0
    for node in route:
        load += demands[node]
    if load > capacity:
        return False
    current_time = 0.0
    prev = 0
    for node in route:
        current_time += dist[prev, node]
        if current_time < ready[node]:
            current_time = ready[node]
        if current_time > due[node]:
            return False
        current_time += service[node]
        prev = node
    return True
 
 
class Plan:
    __slots__ = ("routes", "inst", "_cost", "_ok", "algo")
 
    def __init__(self, routes: List[List[int]], inst: Inst, algo: str = ""):
        self.routes = [r for r in routes if r]
        self.inst   = inst
        self._cost: Optional[float] = None
        self._ok:   Optional[bool]  = None
        self.algo   = algo
 
    @property
    def cost(self) -> float:
        if self._cost is None:
            self._cost = sum(
                _route_cost(np.array(r, np.int64), self.inst.dist)
                for r in self.routes
            )
        return self._cost
 
    @property
    def feasible(self) -> bool:
        if self._ok is None:
            self._ok = all(
                _route_ok(np.array(r, np.int64), self.inst.demands,
                          self.inst.capacity, self.inst.ready_times,
                          self.inst.due_times, self.inst.service_times,
                          self.inst.dist)
                for r in self.routes
            )
        return self._ok
 
    @property
    def nv(self) -> int:
        return len(self.routes)
 
    def dominates(self, other: "Plan") -> bool:
        return self.nv < other.nv or (self.nv == other.nv and self.cost < other.cost)
 
    def copy(self) -> "Plan":
        return Plan([r[:] for r in self.routes], self.inst, self.algo)
 
    def invalidate(self) -> None:
        self._cost = None
        self._ok   = None
 
    def gap(self) -> Tuple[Optional[float], Optional[int]]:
        bks = BKS.get(self.inst.name)
        if not bks:
            return None, None
        return (self.cost - bks["td"]) / bks["td"] * 100, self.nv - bks["nv"]
 
    @property
    def on_time_rate(self) -> float:
        on_time = 0
        total   = 0
        for route in self.routes:
            current_time, prev = 0.0, 0
            for node in route:
                current_time += self.inst.dist[prev, node]
                current_time  = max(current_time, self.inst.ready_times[node])
                total        += 1
                if current_time <= self.inst.due_times[node]:
                    on_time += 1
                current_time += self.inst.service_times[node]
                prev = node
        return on_time / max(total, 1)
 
 
DATASETS = load_datasets(CFG.data_path)
RC1, RC2  = DATASETS.get("rc1", []), DATASETS.get("rc2", [])
print('✅ Data & Solution model ready.')
 

✅ Data & Solution model ready.


In [3]:
# ── Cell 5: ALNS Operators ────────────────────────────────────────────────────
def _invalidate(plan: Plan) -> Plan:
    plan.invalidate()
    return plan
 
 
def _check_route(route: List[int], inst: Inst) -> bool:
    return bool(_route_ok(
        np.array(route, np.int64), inst.demands, inst.capacity,
        inst.ready_times, inst.due_times, inst.service_times, inst.dist,
    ))
 
 
def _best_insert_position(node: int, route: List[int],
                          inst: Inst) -> Tuple[float, Optional[int]]:
    best_cost, best_pos = float("inf"), None
    for pos in range(len(route) + 1):
        prev  = route[pos - 1] if pos > 0 else 0
        nxt   = route[pos]     if pos < len(route) else 0
        delta = inst.dist[prev, node] + inst.dist[node, nxt] - inst.dist[prev, nxt]
        if delta < best_cost and _check_route(route[:pos] + [node] + route[pos:], inst):
            best_cost, best_pos = delta, pos
    return best_cost, best_pos
 
 
def _insert_customer(plan: Plan, node: int, inst: Inst) -> None:
    best_cost, best_route, best_pos = float("inf"), None, None
    for ri, route in enumerate(plan.routes):
        delta, pos = _best_insert_position(node, route, inst)
        if pos is not None and delta < best_cost:
            best_cost, best_route, best_pos = delta, ri, pos
    if best_route is not None:
        plan.routes[best_route].insert(best_pos, node)
    else:
        plan.routes.append([node])
    plan.invalidate()
 
 
def op_random(plan: Plan, size: int) -> Tuple[Plan, List[int]]:
    nodes   = [n for r in plan.routes for n in r]
    removed = random.sample(nodes, min(size, len(nodes)))
    rs      = set(removed)
    plan.routes = [[n for n in r if n not in rs] for r in plan.routes]
    plan.routes = [r for r in plan.routes if r]
    return _invalidate(plan), removed
 
 
def op_worst(plan: Plan, size: int) -> Tuple[Plan, List[int]]:
    inst  = plan.inst
    gains: List[Tuple[float, int]] = []
    for route in plan.routes:
        for idx, node in enumerate(route):
            prev = route[idx - 1] if idx > 0 else 0
            nxt  = route[idx + 1] if idx < len(route) - 1 else 0
            gains.append((
                inst.dist[prev, node] + inst.dist[node, nxt] - inst.dist[prev, nxt],
                node,
            ))
    gains.sort(reverse=True)
    removed = [n for _, n in gains[:size]]
    rs      = set(removed)
    plan.routes = [[n for n in r if n not in rs] for r in plan.routes]
    plan.routes = [r for r in plan.routes if r]
    return _invalidate(plan), removed
 
 
def op_shaw(plan: Plan, size: int) -> Tuple[Plan, List[int]]:
    inst  = plan.inst
    nodes = [n for r in plan.routes for n in r]
    if not nodes:
        return plan, []
    seed    = random.choice(nodes)
    removed = [seed]
    rs      = {seed}
    max_dist = inst.max_dist + 1e-9
    max_tw   = max(inst.due_times - inst.ready_times) + 1e-9
    while len(removed) < size:
        candidates = [
            (n,
             0.5 * inst.dist[seed, n] / max_dist
             + 0.4 * abs(inst.ready_times[seed] - inst.ready_times[n]) / max_tw
             + 0.1 * abs(inst.demands[seed] - inst.demands[n]) / inst.capacity)
            for n in nodes if n not in rs
        ]
        if not candidates:
            break
        nxt = min(candidates, key=lambda x: x[1])[0]
        removed.append(nxt)
        rs.add(nxt)
    plan.routes = [[n for n in r if n not in rs] for r in plan.routes]
    plan.routes = [r for r in plan.routes if r]
    return _invalidate(plan), removed
 
 
def op_route(plan: Plan, size: int) -> Tuple[Plan, List[int]]:
    if len(plan.routes) <= 1:
        return op_random(plan, size)
    removed: List[int] = []
    route_ids: set     = set()
    for idx, route in sorted(enumerate(plan.routes), key=lambda x: len(x[1])):
        if len(removed) + len(route) <= size * 1.5:
            removed.extend(route)
            route_ids.add(idx)
        if len(removed) >= size:
            break
    plan.routes = [r for idx, r in enumerate(plan.routes) if idx not in route_ids] or [[]]
    return _invalidate(plan), removed
 
 
def op_tw_urgent(plan: Plan, size: int) -> Tuple[Plan, List[int]]:
    inst  = plan.inst
    nodes = [n for r in plan.routes for n in r]
    if not nodes:
        return plan, []
    removed = sorted(nodes, key=lambda n: inst.due_times[n] - inst.ready_times[n])[:size]
    rs      = set(removed)
    plan.routes = [[n for n in r if n not in rs] for r in plan.routes]
    plan.routes = [r for r in plan.routes if r]
    return _invalidate(plan), removed
 
 
def op_greedy(plan: Plan, removed: List[int]) -> Plan:
    inst = plan.inst
    for node in sorted(removed, key=lambda n: inst.due_times[n]):
        _insert_customer(plan, node, inst)
    return Plan(plan.routes, inst, plan.algo)
 
 
def _regret(plan: Plan, removed: List[int], k: int) -> Plan:
    inst      = plan.inst
    remaining = removed[:]
    while remaining:
        best_regret, chosen, choice = -float("inf"), None, None
        for node in remaining:
            options = sorted(
                (delta, ri, pos)
                for ri, route in enumerate(plan.routes)
                for delta, pos in [_best_insert_position(node, route, inst)]
                if pos is not None
            )
            if not options:
                continue
            if len(options) >= k:
                regret = sum(options[i][0] - options[0][0] for i in range(1, k))
            elif len(options) >= 2:
                regret = options[1][0] - options[0][0]
            else:
                regret = float("inf")
            if regret > best_regret:
                best_regret, chosen, choice = regret, node, options[0]
        if chosen is not None and choice is not None:
            _, ri, pos = choice
            plan.routes[ri].insert(pos, chosen)
            plan.invalidate()
            remaining.remove(chosen)
        else:
            for node in remaining:
                plan.routes.append([node])
            break
    return Plan(plan.routes, inst, plan.algo)
 
 
def op_regret_2(plan: Plan, removed: List[int]) -> Plan:
    return _regret(plan, removed, 2)
 
 
def op_regret_3(plan: Plan, removed: List[int]) -> Plan:
    return _regret(plan, removed, 3)
 
 
def op_tw_greedy(plan: Plan, removed: List[int]) -> Plan:
    inst = plan.inst
    for node in sorted(removed, key=lambda n: inst.due_times[n] - inst.ready_times[n]):
        _insert_customer(plan, node, inst)
    return Plan(plan.routes, inst, plan.algo)
 
 
DESTROY = [op_random, op_worst, op_shaw, op_route, op_tw_urgent]
REPAIR  = [op_greedy, op_regret_2, op_regret_3, op_tw_greedy]
N_D, N_R = len(DESTROY), len(REPAIR)
print(f'✅ Operators: {N_D}D × {N_R}R = {N_D * N_R} combos')
 
 

✅ Operators: 5D × 4R = 20 combos


In [4]:
# ── Cell 6: Helpers ───────────────────────────────────────────────────────────
def _roulette(weights: np.ndarray) -> int:
    return int(np.random.choice(len(weights), p=weights / weights.sum()))
 
 
def _avg_slack(plan: Plan) -> float:
    inst       = plan.inst
    slack_sum  = 0.0
    count      = 0
    for route in plan.routes:
        current_time, prev = 0.0, 0
        for node in route:
            current_time += inst.dist[prev, node]
            current_time  = max(current_time, inst.ready_times[node])
            slack_sum    += inst.due_times[node] - current_time
            current_time += inst.service_times[node]
            prev          = node
            count        += 1
    return (slack_sum / count) / max(inst.horizon, 1) if count else 0.0
 
 
def _plan_spread(plan: Plan, inst: Inst) -> Tuple[float, float]:
    lengths = [len(r) for r in plan.routes] or [0]
    loads   = [sum(inst.demands[n] for n in r) for r in plan.routes] or [0]
    rb = float(np.std(lengths) / max(np.mean(lengths), 1)) if len(lengths) > 1 else 0.0
    lb = float(np.std(loads)   / max(inst.capacity, 1))
    return min(rb, 1.0), min(lb, 1.0)
 
 
def accept(cur: Plan, cand: Plan, temp: float) -> bool:
    if not cand.feasible:
        return False
    if cand.nv < cur.nv:
        return True
    if cand.nv == cur.nv:
        if cand.cost < cur.cost:
            return True
        return random.random() < math.exp(-(cand.cost - cur.cost) / max(temp, 1e-6))
    return False
 
 
def destroy_size(it: int, n_iters: int, cfg: Config,
                 n_customers: int, scale: float = 1.0) -> int:
    ratio = cfg.destroy_ratio_max - (
        (cfg.destroy_ratio_max - cfg.destroy_ratio_min) * (it / max(n_iters, 1))
    )
    ratio = min(cfg.destroy_ratio_max, max(cfg.destroy_ratio_min, ratio * scale))
    return max(3, int(ratio * n_customers))
 
 
def build_greedy(inst: Inst, algo: str = "") -> Plan:
    """Greedy construction heuristic with feasibility fallback."""
    def arrival(route, pos, node, arrivals):
        prev = route[pos - 1] if pos > 0 else 0
        t    = arrivals[pos - 1] if pos > 0 else 0.0
        t   += inst.dist[prev, node]
        return max(t, inst.ready_times[node])
 
    def feasible_insert(route, pos, node, arrivals, load):
        if load + inst.demands[node] > inst.capacity:
            return False, None
        t = arrival(route, pos, node, arrivals)
        if t > inst.due_times[node]:
            return False, None
        ft   = t + inst.service_times[node]
        prev = node
        for idx in range(pos, len(route)):
            nn  = route[idx]
            ft += inst.dist[prev, nn]
            ft  = max(ft, inst.ready_times[nn])
            if ft > inst.due_times[nn]:
                return False, None
            ft  += inst.service_times[nn]
            prev = nn
        return True, t
 
    def compute_arrivals(route):
        arr: List[float] = []
        t, prev = 0.0, 0
        for node in route:
            t += inst.dist[prev, node]
            t  = max(t, inst.ready_times[node])
            arr.append(t)
            t   += inst.service_times[node]
            prev = node
        return arr
 
    def best_insert_cost(route, node, arrivals, load):
        best_cost, best_pos = float("inf"), None
        for pos in range(len(route) + 1):
            ok, _ = feasible_insert(route, pos, node, arrivals, load)
            if not ok:
                continue
            prev  = route[pos - 1] if pos > 0 else 0
            nxt   = route[pos]     if pos < len(route) else 0
            delta = inst.dist[prev, node] + inst.dist[node, nxt] - inst.dist[prev, nxt]
            if delta < best_cost:
                best_cost, best_pos = delta, pos
        return best_cost, best_pos
 
    unrouted = list(range(1, inst.n + 1))
    routes: List[List[int]] = []
    while unrouted:
        seed = max(unrouted, key=lambda n: inst.dist[0, n])
        if max(inst.dist[0, seed], inst.ready_times[seed]) > inst.due_times[seed]:
            seed = min(unrouted, key=lambda n: inst.due_times[n])
        route    = [seed]
        load     = inst.demands[seed]
        arrivals = [max(inst.dist[0, seed], inst.ready_times[seed])]
        unrouted.remove(seed)
        improved = True
        while improved and unrouted:
            improved = False
            best_reg, best_node, best_pos = -float("inf"), None, None
            for node in unrouted:
                c1, pos = best_insert_cost(route, node, arrivals, load)
                if pos is None:
                    continue
                c2 = inst.dist[0, node] + inst.dist[node, 0] - c1
                if c2 > best_reg:
                    best_reg, best_node, best_pos = c2, node, pos
            if best_node is not None:
                route.insert(best_pos, best_node)
                load    += inst.demands[best_node]
                arrivals = compute_arrivals(route)
                unrouted.remove(best_node)
                improved = True
        routes.append(route)
 
    plan = Plan(routes, inst, algo)
    if plan.feasible:
        return plan
 
    # Fallback: time-window sorted insertion
    customers    = sorted(range(1, inst.n + 1),
                          key=lambda n: (inst.due_times[n], inst.ready_times[n]))
    unrouted_set = set(customers)
    fallback:    List[List[int]] = []
    while unrouted_set:
        route: List[int] = []
        node = 0; load = 0.0; t = 0.0
        while unrouted_set:
            feasible = [
                c for c in unrouted_set
                if load + inst.demands[c] <= inst.capacity
                and t + inst.dist[node, c] <= inst.due_times[c]
            ]
            if not feasible:
                break
            nxt  = min(feasible, key=lambda c: inst.dist[node, c])
            route.append(nxt)
            unrouted_set.remove(nxt)
            load += inst.demands[nxt]
            t     = (max(t + inst.dist[node, nxt], inst.ready_times[nxt])
                     + inst.service_times[nxt])
            node  = nxt
        if route:
            fallback.append(route)
        elif unrouted_set:
            nxt = next(iter(unrouted_set))
            fallback.append([nxt])
            unrouted_set.remove(nxt)
    return Plan(fallback, inst, algo)
 
 
print('✅ Helpers ready.')
 

✅ Helpers ready.


In [5]:
# ── Cell 7: Neural Networks ────────────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buf: Deque[Tuple] = deque(maxlen=capacity)
 
    def push(self, *transition) -> None:
        self.buf.append(transition)
 
    def sample(self, batch_size: int):
        s, a, r, ns, d = zip(*random.sample(self.buf, batch_size))
        return (np.array(s,  np.float32), np.array(a,  np.int64),
                np.array(r,  np.float32), np.array(ns, np.float32),
                np.array(d,  np.float32))
 
    def __len__(self) -> int:
        return len(self.buf)
 
 
class QNet(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
 
 
print('✅ QNet ready.')
 


✅ QNet ready.


In [6]:
# ── Cell 8: ALNS Solver ───────────────────────────────────────────────────────
class ALNSSolver:
    def __init__(self, inst: Inst, cfg: Config):
        self.inst = inst
        self.cfg  = cfg
 
    def solve(self, seed: Optional[int] = None,
              init: Optional[Plan] = None) -> Tuple[Plan, List[float]]:
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)
        cfg  = self.cfg
        cur  = init.copy() if init else build_greedy(self.inst, "ALNS")
        best = cur.copy()
        temp = cfg.temp_control * cur.cost / math.log(2)
        dw   = np.ones(N_D)
        rw   = np.ones(N_R)
        seg_scores = np.zeros((N_D, N_R))
        seg_counts = np.zeros((N_D, N_R))
        history    = [best.cost]
        no_imp     = 0
 
        for it in range(cfg.alns_iterations):
            di   = _roulette(dw)
            ri   = _roulette(rw)
            size = destroy_size(it, cfg.alns_iterations, cfg, self.inst.n)
            dest, removed = DESTROY[di](cur.copy(), size)
            cand          = REPAIR[ri](dest, removed)
            score         = 0
            if accept(cur, cand, temp):
                if cand.dominates(best):
                    best   = cand.copy()
                    score  = cfg.sigma1
                    no_imp = 0
                elif cand.dominates(cur):
                    score  = cfg.sigma2
                    no_imp = 0
                else:
                    score   = cfg.sigma3
                    no_imp += 1
                cur = cand
            else:
                no_imp += 1
 
            seg_scores[di, ri] += score
            seg_counts[di, ri] += 1
 
            if (it + 1) % cfg.segment_size == 0:
                for d in range(N_D):
                    for r in range(N_R):
                        if seg_counts[d, r] > 0:
                            avg    = seg_scores[d, r] / seg_counts[d, r]
                            dw[d]  = dw[d] * (1 - cfg.weight_decay) + avg * cfg.weight_decay
                            rw[r]  = rw[r] * (1 - cfg.weight_decay) + avg * cfg.weight_decay
                seg_scores[:] = 0
                seg_counts[:] = 0
                dw = np.maximum(dw, 0.1)
                rw = np.maximum(rw, 0.1)
 
            temp *= cfg.temp_decay
            history.append(best.cost)
            if no_imp >= cfg.early_stop_patience:
                break
 
        best.algo = "ALNS"
        return best, history
 
 
print('✅ ALNS ready.')
 

✅ ALNS ready.


In [7]:

# ── Cell 9: DQN Solver (ablation only) ────────────────────────────────────────
class DQNNet(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
 
 
class DQNSolver:
    """Pure constructive RL — ablation study only. Not competitive."""
 
    def __init__(self, inst: Inst, cfg: Config = CFG):
        self.inst = inst
        self.cfg  = cfg
        self.q    = DQNNet(cfg.dqn_state_dim, inst.n + 1, cfg.dqn_hidden).to(DEVICE)
        self.q_t  = DQNNet(cfg.dqn_state_dim, inst.n + 1, cfg.dqn_hidden).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt  = optim.Adam(self.q.parameters(), lr=cfg.dqn_lr)
        self.buf  = ReplayBuffer(cfg.dqn_buffer)
        self.eps  = cfg.dqn_eps_start
 
    def _state(self, node, visited, load, t):
        inst = self.inst
        uv   = inst.n - len(visited)
        feas = [n for n in range(1, inst.n + 1)
                if n not in visited
                and load + inst.demands[n] <= inst.capacity
                and t + inst.dist[node, n] <= inst.due_times[n]]
        nf   = len(feas)
        if feas:
            slacks = [inst.due_times[n] - (t + inst.dist[node, n]) for n in feas]
            ms = min(slacks) / max(inst.horizon, 1)
            av = (sum(slacks) / nf) / max(inst.horizon, 1)
            uf = sum(1 for s in slacks if s < 0.1 * inst.horizon) / max(nf, 1)
            aw = (sum(inst.tw_width[n] for n in feas) / nf) / max(inst.max_tw_width, 1)
        else:
            ms = av = uf = aw = 0.0
        return np.array([
            load / inst.capacity, t / max(inst.horizon, 1),
            len(visited) / inst.n, (inst.capacity - load) / inst.capacity,
            uv / inst.n, nf / max(uv, 1),
            inst.coords[node, 0] / 100, inst.coords[node, 1] / 100,
            inst.demands[node] / inst.capacity,
            ms, av, uf, aw,
        ], dtype=np.float32)
 
    def _acts(self, node, visited, load, t):
        inst = self.inst
        acts = [0]
        for n in range(1, inst.n + 1):
            if (n not in visited
                    and load + inst.demands[n] <= inst.capacity
                    and t + inst.dist[node, n] <= inst.due_times[n]):
                acts.append(n)
        return acts
 
    def _sel(self, state, feasible):
        if random.random() < self.eps:
            return random.choice(feasible)
        with torch.no_grad():
            q = self.q(torch.tensor(state).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
        return max(feasible, key=lambda a: q[a])
 
    def _train(self):
        if len(self.buf) < self.cfg.dqn_batch:
            return
        s, a, r, ns, d = self.buf.sample(self.cfg.dqn_batch)
        s  = torch.tensor(s).to(DEVICE)
        a  = torch.tensor(a, dtype=torch.long).to(DEVICE)
        r  = torch.tensor(r).to(DEVICE)
        ns = torch.tensor(ns).to(DEVICE)
        d  = torch.tensor(d).to(DEVICE)
        qp = self.q(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            tgt = r + self.cfg.dqn_gamma * self.q_t(ns).max(1)[0] * (1 - d)
        loss = F.mse_loss(qp, tgt)
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 1.0)
        self.opt.step()
 
    def _episode(self):
        inst    = self.inst
        visited: set         = set()
        routes: List[List[int]] = []
        trans:  List         = []
        while len(visited) < inst.n:
            route: List[int] = []
            node = 0; load = 0.0; t = 0.0; is_new = True
            while True:
                state = self._state(node, visited, load, t)
                feas  = self._acts(node, visited, load, t)
                if len(feas) == 1:
                    break
                action = self._sel(state, feas)
                if action == 0:
                    break
                dv  = inst.dist[node, action]
                rew = -dv / max(inst.max_dist, 1)
                if is_new and routes:
                    rew -= self.cfg.dqn_vehicle_penalty / inst.n
                is_new  = False
                load   += inst.demands[action]
                t       = max(t + dv, inst.ready_times[action]) + inst.service_times[action]
                visited.add(action)
                route.append(action)
                ns   = self._state(action, visited, load, t)
                done = float(len(visited) == inst.n)
                trans.append((state, action, rew, ns, done))
                node = action
            if route:
                routes.append(route)
        return Plan(routes, inst, "DQN"), trans
 
    def solve(self, seed: Optional[int] = None) -> Tuple[Plan, List[float]]:
        if seed is not None:
            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        cfg  = self.cfg
        best = None
        bc   = float("inf")
        hist: List[float] = []
        self.eps = cfg.dqn_eps_start
        n_eps    = max(50, cfg.alns_iterations // self.inst.n)
        for ep in range(n_eps):
            plan, trans = self._episode()
            if plan.feasible and trans:
                bonus    = max(0, (bc - plan.cost) / bc * 10) if bc < float("inf") else 1.0
                s, a, r, ns, d = trans[-1]
                trans[-1] = (s, a, r + bonus, ns, d)
                if plan.cost < bc:
                    bc   = plan.cost
                    best = plan.copy()
            for tr in trans:
                self.buf.push(*tr)
            if ep % cfg.dqn_train_freq == 0:
                for _ in range(min(5, len(self.buf) // max(cfg.dqn_batch, 1))):
                    self._train()
            if ep % cfg.dqn_target_freq == 0:
                self.q_t.load_state_dict(self.q.state_dict())
            self.eps = max(cfg.dqn_eps_end, self.eps * cfg.dqn_eps_decay)
            hist.append(bc if bc < float("inf") else float("nan"))
        if best is None:
            best = build_greedy(self.inst, "DQN")
        best.algo = "DQN"
        return best, hist
 
 
print('✅ DQN solver ready.')

✅ DQN solver ready.


In [8]:
# ── Cell 10: PlateauHybridSolver (DDQN-ALNS) ─────────────────────────────────
class PlateauController:
    def __init__(self, cfg: Config):
        self.cfg  = cfg
        self.q    = QNet(cfg.ctrl_state_dim, len(MODES), cfg.ctrl_hidden).to(DEVICE)
        self.q_t  = QNet(cfg.ctrl_state_dim, len(MODES), cfg.ctrl_hidden).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt  = optim.Adam(self.q.parameters(), lr=cfg.ctrl_lr)
        self.buf  = ReplayBuffer(cfg.ctrl_buffer)
        self.eps  = cfg.ctrl_eps_start
        self.step = 0
 
    def reset(self) -> None:
        self.eps = self.cfg.ctrl_eps_start
 
    def act(self, state: np.ndarray, force_default: bool = False) -> int:
        if force_default:
            return MODE_DEFAULT
        if random.random() < self.eps:
            return random.randrange(len(MODES))
        with torch.no_grad():
            return int(
                self.q(torch.tensor(state).unsqueeze(0).to(DEVICE))[0].argmax().item()
            )
 
    def observe(self, state: np.ndarray, action: int, reward: float,
                next_state: np.ndarray, done: float = 0.0) -> None:
        self.buf.push(state, action, reward, next_state, done)
 
    def train_step(self) -> None:
        self.step += 1
        if len(self.buf) < self.cfg.ctrl_batch:
            return
        s, a, r, ns, d = self.buf.sample(self.cfg.ctrl_batch)
        s  = torch.tensor(s).to(DEVICE)
        a  = torch.tensor(a, dtype=torch.long).to(DEVICE)
        r  = torch.tensor(r).to(DEVICE)
        ns = torch.tensor(ns).to(DEVICE)
        d  = torch.tensor(d).to(DEVICE)
        qp = self.q(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            target = (r + self.cfg.ctrl_gamma
                      * self.q_t(ns).gather(1, self.q(ns).argmax(1).unsqueeze(1)).squeeze(1)
                      * (1 - d))
        loss = F.mse_loss(qp, target)
        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 1.0)
        self.opt.step()
        if self.step % self.cfg.ctrl_target_freq == 0:
            self.q_t.load_state_dict(self.q.state_dict())
        self.eps = max(self.cfg.ctrl_eps_end, self.eps * self.cfg.ctrl_eps_decay)
 
 
class PlateauHybridSolver:
    """
    DDQN-ALNS (v9.5):
    PlateauController selects search MODE (default/intensify/diversify/tw_rescue)
    when a plateau is detected.  NV-increase is now penalised in the reward.
    """
 
    def __init__(self, inst: Inst, cfg: Config):
        self.inst    = inst
        self.cfg     = cfg
        self.ctrl    = PlateauController(cfg)
        self.op_counts: Dict[Tuple[int, int], int] = {}  # for export
 
    # ── [FIX-3] clone_weights ─────────────────────────────────────────────────
    def clone_weights(self) -> Dict:
        return {k: v.clone().cpu() for k, v in self.ctrl.q.state_dict().items()}
 
    def load_weights(self, weights: Dict) -> None:
        self.ctrl.q.load_state_dict(weights)
        self.ctrl.q_t.load_state_dict(weights)
 
    def _state(self, cur: Plan, best: Plan, no_imp: int, temp: float,
               dw: np.ndarray, rw: np.ndarray,
               imp_rate: float, progress: float) -> np.ndarray:
        rb, lb = _plan_spread(cur, self.inst)
        t0     = self.cfg.temp_control * best.cost / math.log(2)
        return np.array([
            min(no_imp / max(self.cfg.early_stop_patience, 1), 1.0),
            min((cur.cost - best.cost) / max(best.cost, 1), 1.0),
            min(temp / max(t0, 1e-6), 1.5),
            imp_rate,
            min(cur.nv / max(self._init_nv, 1), 2.0),
            rb, lb,
            self.inst.tw_tight_frac,
            _avg_slack(cur),
            progress,
        ], dtype=np.float32)
 
    def _segment_reward(self, best_before: Plan, best_after: Plan,
                        cur_before: Plan, cur_after: Plan,
                        accepted_moves: int) -> float:
        reward = -0.75
 
        if best_after.nv < best_before.nv:
            # Vehicle reduction: large positive reward
            reward += (25.0 * (best_before.nv - best_after.nv)
                       + max((best_before.cost - best_after.cost)
                             / max(best_before.cost, 1), 0.0) * 100)
 
        elif best_after.nv > best_before.nv:
            # [PERF-1] NV increase: penalise to stop cost/vehicle arbitrage
            reward -= self.cfg.nv_increase_penalty * (best_after.nv - best_before.nv)
 
        elif best_after.cost < best_before.cost:
            # Same NV, better cost
            reward += (2.5 * (best_before.cost - best_after.cost)
                       / max(best_before.cost, 1) * 100)
 
        if (cur_after.nv == cur_before.nv
                and cur_after.cost < cur_before.cost):
            reward += (0.5 * (cur_before.cost - cur_after.cost)
                       / max(cur_before.cost, 1) * 100)
 
        if accepted_moves == 0:
            reward -= 0.5
 
        return float(reward)
 
    def solve(self, seed: Optional[int] = None,
              frozen: bool = False) -> Tuple[Plan, List[float]]:
        """
        frozen=True: weights are pre-loaded (transfer); controller does not train.
        """
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
 
        cfg  = self.cfg
        self.ctrl.reset()
        self.op_counts = {}
 
        cur  = build_greedy(self.inst, "DDQN-ALNS")
        best = cur.copy()
        self._init_nv = cur.nv
 
        temp = cfg.temp_control * cur.cost / math.log(2)
        dw   = np.ones(N_D)
        rw   = np.ones(N_R)
        history: List[float]        = [best.cost]
        recent_improvements: Deque  = deque(maxlen=cfg.segment_size)
        no_imp       = 0
        post_imp_lock = 0
        n_segments   = math.ceil(cfg.hybrid_iterations / cfg.segment_size)
 
        for seg_idx in range(n_segments):
            progress = seg_idx / max(n_segments, 1)
            imp_rate = (sum(recent_improvements) / len(recent_improvements)
                        if recent_improvements else 0.0)
            state_before = self._state(cur, best, no_imp, temp, dw, rw,
                                       imp_rate, progress)
 
            if post_imp_lock > 0:
                action = MODE_INTENSIFY
                post_imp_lock -= 1
                ctrl_active = False
            elif no_imp >= cfg.plateau_start:
                action      = self.ctrl.act(state_before, force_default=frozen)
                ctrl_active = not frozen
            else:
                action      = MODE_DEFAULT
                ctrl_active = False
 
            mode      = MODES[action]
            biased_dw = np.maximum(dw * np.array(mode.destroy_bias, np.float32), 0.1)
            biased_rw = np.maximum(rw * np.array(mode.repair_bias,  np.float32), 0.1)
            temp     *= mode.temp_boost
 
            seg_scores    = np.zeros((N_D, N_R))
            seg_counts    = np.zeros((N_D, N_R))
            seg_best_pre  = best.copy()
            seg_cur_pre   = cur.copy()
            accepted_moves = 0
            best_improved  = False
 
            for offset in range(cfg.segment_size):
                it = seg_idx * cfg.segment_size + offset
                if it >= cfg.hybrid_iterations:
                    break
 
                di   = _roulette(biased_dw)
                ri   = _roulette(biased_rw)
                size = destroy_size(it, cfg.hybrid_iterations, cfg,
                                    self.inst.n, scale=mode.destroy_scale)
                dest, removed = DESTROY[di](cur.copy(), size)
                cand          = REPAIR[ri](dest, removed)
                score         = 0
                improved      = False
 
                if accept(cur, cand, temp):
                    accepted_moves += 1
                    improved        = cand.dominates(cur)
                    if cand.dominates(best):
                        best          = cand.copy()
                        best_improved = True
                        score         = cfg.sigma1
                        no_imp        = 0
                    elif improved:
                        score  = cfg.sigma2
                        no_imp = 0
                    else:
                        score   = cfg.sigma3
                        no_imp += 1
                    cur = cand
                else:
                    no_imp += 1
 
                # Track operator usage for export
                key = (di, ri)
                self.op_counts[key] = self.op_counts.get(key, 0) + 1
 
                recent_improvements.append(1 if improved else 0)
                seg_scores[di, ri] += score
                seg_counts[di, ri] += 1
                temp *= cfg.temp_decay * mode.temp_decay_scale
                history.append(best.cost)
 
                if no_imp >= cfg.early_stop_patience:
                    break
 
            # Update adaptive weights
            for d in range(N_D):
                for r in range(N_R):
                    if seg_counts[d, r] > 0:
                        avg   = seg_scores[d, r] / seg_counts[d, r]
                        dw[d] = dw[d] * (1 - cfg.weight_decay) + avg * cfg.weight_decay
                        rw[r] = rw[r] * (1 - cfg.weight_decay) + avg * cfg.weight_decay
            dw = np.maximum(dw, 0.1)
            rw = np.maximum(rw, 0.1)
 
            # Controller observe & train
            state_after = self._state(
                cur, best, no_imp, temp, dw, rw,
                sum(recent_improvements) / len(recent_improvements)
                if recent_improvements else 0.0,
                min((seg_idx + 1) / max(n_segments, 1), 1.0),
            )
            if ctrl_active:
                self.ctrl.observe(
                    state_before, action,
                    self._segment_reward(seg_best_pre, best,
                                         seg_cur_pre, cur, accepted_moves),
                    state_after, 0.0,
                )
                self.ctrl.train_step()
 
            if best_improved:
                post_imp_lock = cfg.post_improve_intensify_segments
 
            if no_imp >= cfg.early_stop_patience:
                break
 
        best.algo = "DDQN-ALNS"
        return best, history
 
 
# [FIX-2] Alias — all downstream cells use RLALNSSolver
RLALNSSolver = PlateauHybridSolver
 
print('✅ PlateauHybridSolver (DDQN-ALNS) ready.')
 

✅ PlateauHybridSolver (DDQN-ALNS) ready.


In [9]:
# ── Cell 11: Benchmark Runner ─────────────────────────────────────────────────
def run_instance(inst: Inst, algo: str, cfg: Config,
                 seed: int,
                 transfer_weights: Optional[Dict] = None) -> Dict:
    start = time.time()
 
    if algo == "ALNS":
        plan, history = ALNSSolver(inst, cfg).solve(seed=seed)
 
    elif algo in ("DDQN-ALNS", "PLATEAU-HYBRID"):
        solver = PlateauHybridSolver(inst, cfg)
        plan, history = solver.solve(seed=seed)
 
    elif algo == "DDQN-ALNS★":
        solver = PlateauHybridSolver(inst, cfg)
        if transfer_weights is not None:
            solver.load_weights(transfer_weights)
        plan, history = solver.solve(seed=seed, frozen=True)
 
    elif algo == "DQN":
        plan, history = DQNSolver(inst, cfg).solve(seed=seed)
 
    else:
        raise ValueError(f"Unsupported algorithm: {algo}")
 
    bks = BKS.get(inst.name)
    return {
        "nv":      plan.nv,
        "cost":    plan.cost,
        "time":    time.time() - start,
        "td_gap":  (plan.cost - bks["td"]) / bks["td"] * 100 if bks else None,
        "nv_diff": plan.nv - bks["nv"] if bks else None,
        "on_time": plan.on_time_rate,
        "hist":    history,
    }
 
 
def run_benchmark(instances:        Iterable[Inst],
                  algorithms:       List[str],
                  cfg:              Config,
                  result_path:      Optional[str]  = None,
                  transfer_weights: Optional[Dict]  = None) -> pd.DataFrame:
    """[FIX-4] transfer_weights kwarg propagated to run_instance."""
    instances    = list(instances)
    result_path  = result_path or os.path.join(cfg.output_dir, "benchmark_clean.csv")
    rows: List[Dict] = []
 
    total = len(instances) * len(algorithms)
    print(f"Total: {total} combos × {cfg.n_runs} runs\n" + "=" * 60)
 
    for inst in instances:
        dataset = "RC1" if inst.name[2] == "1" else "RC2"
        for algo in algorithms:
            print(f"\n[{inst.name}] {algo}")
            nv_v, cost_v, time_v, gap_v, nvd_v, ot_v = [], [], [], [], [], []
            for run_idx in range(cfg.n_runs):
                res = run_instance(inst, algo, cfg,
                                   cfg.seed + run_idx, transfer_weights)
                nv_v.append(res["nv"])
                cost_v.append(res["cost"])
                time_v.append(res["time"])
                gap_v.append(res["td_gap"])
                nvd_v.append(res["nv_diff"])
                ot_v.append(res["on_time"])
                print(f"  run {run_idx + 1}/{cfg.n_runs}: "
                      f"nv={res['nv']} cost={res['cost']:.1f} ({res['time']:.1f}s)")
 
            row = {
                "Dataset":   dataset,
                "Instance":  inst.name,
                "Algorithm": algo,
                "NV_mean":   round(np.mean(nv_v),   2),
                "NV_std":    round(np.std(nv_v),    2),
                "NV_diff":   round(np.mean(nvd_v),  2) if nvd_v[0] is not None else None,
                "TD_mean":   round(np.mean(cost_v), 2),
                "TD_std":    round(np.std(cost_v),  2),
                "Gap%":      round(np.mean(gap_v),  2) if gap_v[0] is not None else None,
                "OnTime":    round(np.mean(ot_v) * 100, 1),
                "Time_s":    round(np.mean(time_v), 1),
                "NV_cv":     round(np.std(nv_v)   / max(np.mean(nv_v),   1) * 100, 2),
                "TD_cv":     round(np.std(cost_v)  / max(np.mean(cost_v), 1) * 100, 2),
            }
            rows.append(row)
            gap_text = f"{row['Gap%']:+.1f}%" if row["Gap%"] is not None else "--"
            print(f"  → nv={row['NV_mean']:.1f}±{row['NV_std']:.1f}  "
                  f"td={row['TD_mean']:.1f}±{row['TD_std']:.1f}  gap={gap_text}")
 
    df = pd.DataFrame(rows)
    df.to_csv(result_path, index=False)
    return df
 
 
def print_summary_table(df: pd.DataFrame) -> None:
    summary = (
        df.groupby(["Dataset", "Algorithm"])
          .agg(NV=("NV_mean", "mean"), NV_std=("NV_std", "mean"),
               NV_diff=("NV_diff", "mean"), TD=("TD_mean", "mean"),
               TD_std=("TD_std", "mean"), Gap=("Gap%", "mean"),
               OnTime=("OnTime", "mean"), Time=("Time_s", "mean"))
          .round(2).reset_index()
    )
    print("\n" + "-" * 86)
    print(f"{'DS':<4}{'Algorithm':<18}{'NV':>6}{'+/-':>6}{'vsBKS':>8}"
          f"{'TD':>10}{'+/-':>8}{'Gap%':>8}{'OT%':>7}{'Time':>8}")
    print("-" * 86)
    for _, row in summary.iterrows():
        gap    = f"{row['Gap']:+.2f}%" if pd.notna(row["Gap"])    else "--"
        nv_diff = f"{row['NV_diff']:+.2f}" if pd.notna(row["NV_diff"]) else "--"
        print(f"{row['Dataset']:<4}{row['Algorithm']:<18}"
              f"{row['NV']:>6.2f}{row['NV_std']:>6.2f}{nv_diff:>8}"
              f"{row['TD']:>10.2f}{row['TD_std']:>8.2f}{gap:>8}"
              f"{row['OnTime']:>7.1f}{row['Time']:>7.1f}s")
    print("-" * 86)
 
 
print('✅ Benchmark runner ready.')
 

✅ Benchmark runner ready.


In [10]:
# ── Cell 12: Transfer Learning Pipeline ───────────────────────────────────────
def train_transfer_model(instances: List[Inst],
                         cfg: Config = CFG,
                         seed: int   = 42) -> Dict:
    """Train DDQN-ALNS on all instances, return averaged Q-network weights."""
    print('📚 Training transfer model...')
    all_weights = []
    for inst in instances:
        print(f'  Training on {inst.name}...')
        solver = PlateauHybridSolver(inst, cfg)
        plan, _ = solver.solve(seed=seed)
        all_weights.append(solver.clone_weights())
        td, nv = plan.gap()
        print(f'    nv={plan.nv}, gap={td:+.1f}%')
 
    keys     = all_weights[0].keys()
    averaged = {
        k: torch.stack([w[k].float() for w in all_weights]).mean(0)
        for k in keys
    }
 
    # [FIX-1] use cfg.output_dir, not MODEL_DIR
    if SAFETENSORS_OK:
        save_path = os.path.join(cfg.output_dir, 'rl_alns_transfer.safetensors')
        save_file(averaged, save_path)
        print(f'\n✅ Transfer model saved → {save_path}')
    else:
        print('⚠️  safetensors unavailable — transfer model not saved to disk')
 
    return averaged
 
 
def load_transfer_model(cfg: Config = CFG) -> Optional[Dict]:
    # [FIX-1] use cfg.output_dir, not MODEL_DIR
    if not SAFETENSORS_OK:
        return None
    path = os.path.join(cfg.output_dir, 'rl_alns_transfer.safetensors')
    if os.path.exists(path):
        print(f'✅ Transfer model loaded from {path}')
        return load_file(path)
    return None
 
 
print('✅ Transfer learning pipeline ready.')

✅ Transfer learning pipeline ready.


In [11]:
# ── Cell 13: Statistical Tests & Paper Tables ─────────────────────────────────
def wilcoxon_test(df: pd.DataFrame, algo_a: str, algo_b: str,
                  metric: str = 'Gap%',
                  dataset: Optional[str] = None) -> Dict:
    sub = df if dataset is None else df[df['Dataset'] == dataset]
    a   = sub[sub['Algorithm'] == algo_a][metric].dropna().values
    b   = sub[sub['Algorithm'] == algo_b][metric].dropna().values
    n   = min(len(a), len(b))
    a, b = a[:n], b[:n]
    if n < 3:
        return {'stat': None, 'p': None, 'sig': False, 'n': n}
    stat, p = stats.wilcoxon(a, b, alternative='two-sided')
    return {
        'stat': round(stat, 3), 'p': round(p, 4), 'sig': p < 0.05,
        'n': n, 'better': algo_a if a.mean() < b.mean() else algo_b,
    }
 
 
def print_paper_table(df: pd.DataFrame) -> None:
    summary = (
        df.groupby(['Dataset', 'Algorithm'])
          .agg(NV=('NV_mean', 'mean'), NV_std=('NV_std', 'mean'),
               NV_d=('NV_diff', 'mean'),
               TD=('TD_mean', 'mean'), TD_std=('TD_std', 'mean'),
               Gap=('Gap%', 'mean'),
               CV_nv=('NV_cv', 'mean'), CV_td=('TD_cv', 'mean'),
               OT=('OnTime', 'mean'), Time=('Time_s', 'mean'))
          .round(2).reset_index()
    )
    hdr = (f'{"DS":<4}{"Algorithm":<14}{"NV":>6}{"±":>4}{"vsBKS":>8}'
           f'{"TD":>9}{"±":>6}{"Gap%":>7}{"CV_NV":>6}{"CV_TD":>6}'
           f'{"OT%":>6}{"Time":>7}')
    sep = '─' * len(hdr)
    print('\n' + sep)
    print(hdr)
    print(sep)
    prev = ''
    for _, r in summary.iterrows():
        if r['Dataset'] != prev and prev:
            print(sep)
        prev   = r['Dataset']
        nv_d   = f"{r['NV_d']:+.1f}" if pd.notna(r['NV_d']) else '—'
        gap    = f"{r['Gap']:+.1f}%" if pd.notna(r['Gap'])   else '—'
        print(f"{r['Dataset']:<4}{r['Algorithm']:<14}"
              f"{r['NV']:>6.1f}{r['NV_std']:>4.1f}{nv_d:>8}"
              f"{r['TD']:>9.1f}{r['TD_std']:>6.1f}{gap:>7}"
              f"{r['CV_nv']:>6.1f}{r['CV_td']:>6.1f}"
              f"{r['OT']:>6.1f}{r['Time']:>6.1f}s")
    print(sep)
    print('CV = std/mean×100%. Negative Gap%: solution beats BKS distance.')
 
 
def print_stats_table(df: pd.DataFrame) -> None:
    print('\n── Wilcoxon signed-rank (two-sided) ──')
    print(f'{"Comparison":<28}{"DS":<5}{"Metric":<8}'
          f'{"W":>7}{"p":>9}{"Sig":>5}{"Better":>12}')
    print('─' * 70)
    pairs = [('DDQN-ALNS', 'ALNS'), ('DDQN-ALNS★', 'ALNS')]
    for algo_a, algo_b in pairs:
        if algo_a not in df['Algorithm'].values:
            continue
        for ds in ['RC1', 'RC2']:
            for metric in ['Gap%', 'NV_mean']:
                res = wilcoxon_test(df, algo_a, algo_b, metric, ds)
                if res['stat'] is None:
                    continue
                sig = '✅' if res['sig'] else '—'
                print(f'  {algo_a} vs {algo_b:<8}  {ds:<5}{metric:<8}'
                      f'{res["stat"]:>7.1f}{res["p"]:>9.4f}'
                      f'{sig:>5}{res["better"]:>12}')
    print('─' * 70)
    print('✅ = p < 0.05')
 
 
print('✅ Stats & table utilities ready.')

✅ Stats & table utilities ready.


In [12]:
# ── Cell 14: Visualisation ────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')          # headless-safe on Kaggle
import matplotlib.pyplot as plt
 
COLORS = {
    'ALNS':       '#5f5fae',
    'DDQN-ALNS':  '#1d9e75',
    'DDQN-ALNS★': '#e67e22',
    'DQN':        '#e74c3c',
    'PLATEAU-HYBRID': '#1d9e75',   # legacy label compatibility
}
 
 
def plot_dashboard(df: pd.DataFrame) -> None:
    metrics = [
        ('Gap%',    'Distance Gap vs BKS (%)', '↓ lower is better'),
        ('NV_mean', 'Vehicles Used (avg)',      '↓ lower is better'),
        ('TD_cv',   'TD Consistency (CV %)',    '↓ lower = more stable'),
    ]
    algos = [a for a in COLORS if a in df['Algorithm'].values]
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    for ri, ds in enumerate(['RC1', 'RC2']):
        for ci, (met, label, note) in enumerate(metrics):
            ax   = axes[ri][ci]
            sub  = df[df['Dataset'] == ds]
            insts = sub['Instance'].unique()
            x    = np.arange(len(insts))
            w    = 0.8 / max(len(algos), 1)
            for ji, algo in enumerate(algos):
                vals = [sub[sub['Instance'] == i][met].mean() for i in insts]
                ax.bar(x + ji * w, vals, w, label=algo,
                       color=COLORS.get(algo, '#888888'), alpha=0.85,
                       edgecolor='white')
            ax.set_xticks(x + w * (len(algos) - 1) / 2)
            ax.set_xticklabels([i[-3:] for i in insts], fontsize=8)
            ax.set_title(f'{ds} — {label}\n({note})', fontsize=9, fontweight='bold')
            ax.set_ylabel(met, fontsize=8)
            ax.grid(axis='y', alpha=0.3)
            if ri == 0 and ci == 0:
                ax.legend(fontsize=8)
    plt.suptitle('Algorithm Comparison — VRPTW Solomon RC Benchmarks v9.5',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(CFG.output_dir, 'dashboard.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'✅ dashboard.png saved → {out}')
 
 
def plot_convergence_grid(inst: Inst, cfg: Config, seed: int = 42) -> None:
    histories = {}
    for algo, SolverCls in [('ALNS', ALNSSolver),
                             ('DDQN-ALNS', PlateauHybridSolver)]:
        s = SolverCls(inst, cfg)
        _, hist = s.solve(seed=seed)
        histories[algo] = hist
 
    fig, ax = plt.subplots(figsize=(9, 4))
    for algo, hist in histories.items():
        ax.plot(hist, label=algo, color=COLORS.get(algo, '#888'), lw=2, alpha=0.9)
    bks = BKS.get(inst.name, {})
    if bks:
        ax.axhline(bks['td'], color='gray', ls='--', lw=1.2, label='BKS distance')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Best Cost Found')
    ax.set_title(f'Convergence — {inst.name}', fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.2)
    plt.tight_layout()
    out = os.path.join(CFG.output_dir, f'convergence_{inst.name}.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'✅ Convergence plot → {out}')
 
 
def plot_transfer_comparison(df: pd.DataFrame) -> None:
    """[FIX-6] Scatter: DDQN-ALNS★ vs ALNS on RC2 Gap%."""
    rc2  = df[df['Dataset'] == 'RC2']
    alns = rc2[rc2['Algorithm'] == 'ALNS'][['Instance', 'Gap%']].set_index('Instance')
    star = rc2[rc2['Algorithm'] == 'DDQN-ALNS★'][['Instance', 'Gap%']].set_index('Instance')
    common = alns.index.intersection(star.index)
    if common.empty:
        print('⚠️  No DDQN-ALNS★ results to plot yet.')
        return
 
    fig, ax = plt.subplots(figsize=(7, 5))
    x = alns.loc[common, 'Gap%'].values
    y = star.loc[common, 'Gap%'].values
    ax.scatter(x, y, s=80, color=COLORS['DDQN-ALNS★'], zorder=3)
    for inst, xi, yi in zip(common, x, y):
        ax.annotate(inst[-3:], (xi, yi), textcoords='offset points',
                    xytext=(4, 4), fontsize=8)
    lim = [min(x.min(), y.min()) - 1, max(x.max(), y.max()) + 1]
    ax.plot(lim, lim, 'k--', lw=1, alpha=0.5, label='y=x (same performance)')
    ax.set_xlabel('ALNS Gap% (RC2)')
    ax.set_ylabel('DDQN-ALNS★ Gap% (RC2, zero-shot)')
    ax.set_title('Transfer Learning: DDQN-ALNS★ vs ALNS on RC2', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    out = os.path.join(CFG.output_dir, 'transfer_comparison.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'✅ Transfer comparison plot → {out}')
 
 
def plot_routes(plan: Plan, save: bool = True) -> None:
    RCOLS = [
        '#E63946', '#2A9D8F', '#E9C46A', '#264653', '#F4A261',
        '#A8DADC', '#457B9D', '#6A4C93', '#F72585', '#4CC9F0',
        '#80B918', '#FF9F1C', '#8338EC', '#3A86FF', '#CBFF8C',
    ]
    inst = plan.inst
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(*inst.coords[0], s=220, c='black', marker='s', zorder=5)
    ax.annotate('DEPOT', inst.coords[0], fontsize=8,
                ha='center', va='bottom', fontweight='bold')
    for i, route in enumerate(plan.routes):
        col   = RCOLS[i % len(RCOLS)]
        stops = [0] + route + [0]
        xs    = [inst.coords[n, 0] for n in stops]
        ys    = [inst.coords[n, 1] for n in stops]
        ax.plot(xs, ys, '-o', color=col, lw=1.5, ms=4, alpha=0.8, label=f'V{i+1}')
    td, nv = plan.gap()
    g = f' | BKS: TD {td:+.1f}% NV {nv:+d}' if td is not None else ''
    ax.set_title(f'{plan.algo} — {inst.name}  nv={plan.nv}  cost={plan.cost:.1f}{g}',
                 fontweight='bold')
    ax.legend(fontsize=6, ncol=3)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    if save:
        out = os.path.join(CFG.output_dir, f'routes_{plan.algo}_{inst.name}.png')
        plt.savefig(out, dpi=120, bbox_inches='tight')
        plt.close()
        print(f'✅ Route plot → {out}')
    else:
        plt.show()
 
 
print('✅ Visualisation ready.')
 
 

✅ Visualisation ready.


In [13]:
# ── Cell 15: Smoke Test ───────────────────────────────────────────────────────
smoke_cfg = Config(
    alns_iterations=400,
    hybrid_iterations=600,
    early_stop_patience=150,
    n_runs=1,
)
inst = RC1[0]
print(f'Smoke test — {inst.name}\n')
 
for algo, SolverCls in [('ALNS', ALNSSolver),
                         ('DDQN-ALNS', PlateauHybridSolver)]:
    t0        = time.time()
    s         = SolverCls(inst, smoke_cfg)
    plan, _   = s.solve(seed=42)
    el        = time.time() - t0
    td, nv    = plan.gap()
    print(f'  {algo:<18} nv={plan.nv:>3}  cost={plan.cost:>8.1f}  '
          f'BKS TD {td:+.1f}% NV {nv:+d}  ({el:.1f}s)')
 
print('\n✓ Smoke test passed')
plot_convergence_grid(RC1[0], smoke_cfg, seed=42)

Smoke test — RC101

  ALNS               nv= 16  cost=  1781.6  BKS TD +5.0% NV +2  (19.4s)
  DDQN-ALNS          nv= 15  cost=  1720.1  BKS TD +1.4% NV +1  (24.3s)

✓ Smoke test passed
✅ Convergence plot → /kaggle/working/convergence_RC101.png


In [14]:
# ── Cell 16: Phase 1 — Main Benchmark ────────────────────────────────────────
RESULT_PATH = os.path.join(CFG.output_dir, 'benchmark_clean.csv')
 
df = run_benchmark(
    instances  = RC1 + RC2,
    algorithms = ['ALNS', 'DDQN-ALNS'],
    cfg        = CFG,
    result_path= RESULT_PATH,
)
print_summary_table(df)
plot_dashboard(df)
 

Total: 32 combos × 5 runs

[RC101] ALNS
  run 1/5: nv=16 cost=1686.8 (130.4s)
  run 2/5: nv=15 cost=1714.8 (102.7s)
  run 3/5: nv=15 cost=1696.5 (110.3s)
  run 4/5: nv=15 cost=1710.8 (123.3s)
  run 5/5: nv=15 cost=1669.3 (104.2s)
  → nv=15.2±0.4  td=1695.6±16.6  gap=-0.1%

[RC101] DDQN-ALNS
  run 1/5: nv=15 cost=1674.9 (87.1s)
  run 2/5: nv=15 cost=1697.3 (90.4s)
  run 3/5: nv=15 cost=1708.7 (96.0s)
  run 4/5: nv=16 cost=1664.6 (90.7s)
  run 5/5: nv=16 cost=1663.8 (77.8s)
  → nv=15.4±0.5  td=1681.8±18.1  gap=-0.9%

[RC102] ALNS
  run 1/5: nv=13 cost=1517.3 (108.6s)
  run 2/5: nv=14 cost=1507.8 (116.8s)
  run 3/5: nv=13 cost=1505.6 (132.2s)
  run 4/5: nv=14 cost=1511.7 (117.4s)
  run 5/5: nv=14 cost=1522.0 (124.7s)
  → nv=13.6±0.5  td=1512.9±6.1  gap=-2.7%

[RC102] DDQN-ALNS
  run 1/5: nv=13 cost=1498.6 (72.0s)
  run 2/5: nv=14 cost=1495.6 (68.8s)
  run 3/5: nv=14 cost=1516.0 (90.0s)
  run 4/5: nv=14 cost=1491.6 (79.9s)
  run 5/5: nv=13 cost=1551.6 (91.2s)
  → nv=13.6±0.5  td=1510.7±22.

In [15]:
# ── Cell 17: Phase 2 — Transfer Learning ─────────────────────────────────────
RESULT_TRANSFER = os.path.join(CFG.output_dir, 'benchmark_transfer.csv')
 
# Step 1: Train (or load cached)
transfer_weights = load_transfer_model(CFG)
if transfer_weights is None:
    transfer_weights = train_transfer_model(RC1, CFG, seed=CFG.seed)
 
# Step 2: Zero-shot on RC2
df_tr = run_benchmark(
    instances        = RC2,
    algorithms       = ['DDQN-ALNS★'],
    cfg              = CFG,
    result_path      = RESULT_TRANSFER,
    transfer_weights = transfer_weights,
)
 
# Step 3: Compare
df_all = pd.concat([df, df_tr], ignore_index=True)
print_paper_table(df_all[df_all['Dataset'] == 'RC2'])
 

📚 Training transfer model...
  Training on RC101...
    nv=15, gap=-1.3%
  Training on RC102...
    nv=13, gap=-3.6%
  Training on RC103...
    nv=12, gap=+2.8%
  Training on RC104...
    nv=10, gap=+4.8%
  Training on RC105...
    nv=15, gap=-2.7%
  Training on RC106...
    nv=12, gap=+0.3%
  Training on RC107...
    nv=12, gap=+1.9%
  Training on RC108...
    nv=11, gap=+1.3%

✅ Transfer model saved → /kaggle/working/rl_alns_transfer.safetensors
Total: 8 combos × 5 runs

[RC201] DDQN-ALNS★
  run 1/5: nv=4 cost=1457.1 (62.7s)
  run 2/5: nv=4 cost=1459.5 (64.4s)
  run 3/5: nv=4 cost=1494.4 (90.2s)
  run 4/5: nv=4 cost=1446.0 (83.5s)
  run 5/5: nv=4 cost=1484.3 (77.1s)
  → nv=4.0±0.0  td=1468.2±18.1  gap=+4.4%

[RC202] DDQN-ALNS★
  run 1/5: nv=4 cost=1161.8 (67.5s)
  run 2/5: nv=4 cost=1236.5 (67.5s)
  run 3/5: nv=4 cost=1161.3 (69.8s)
  run 4/5: nv=4 cost=1228.1 (81.4s)
  run 5/5: nv=4 cost=1222.4 (74.5s)
  → nv=4.0±0.0  td=1202.0±33.4  gap=-12.0%

[RC203] DDQN-ALNS★
  run 1/5: nv=3 co

In [16]:
# ── Cell 18: Full Analysis ────────────────────────────────────────────────────
dfs = []
for p in [RESULT_PATH, RESULT_TRANSFER]:
    if os.path.exists(p):
        dfs.append(pd.read_csv(p))
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
 
if not df_all.empty:
    print('=' * 60 + '\nFULL RESULTS TABLE')
    print_paper_table(df_all)
 
    print('\n' + '=' * 60 + '\nSTATISTICAL TESTS')
    print_stats_table(df_all)
 
    print('\n' + '=' * 60 + '\nDASHBOARD')
    plot_dashboard(df_all)
 
    print('\n' + '=' * 60 + '\nTRANSFER: DDQN-ALNS★ vs ALNS on RC2')
    plot_transfer_comparison(df_all)
else:
    print('Run cells 16 and 17 first.')
 

FULL RESULTS TABLE

───────────────────────────────────────────────────────────────────────────────────
DS  Algorithm         NV   ±   vsBKS       TD     ±   Gap% CV_NV CV_TD   OT%   Time
───────────────────────────────────────────────────────────────────────────────────
RC1 ALNS            12.5 0.3    +1.0   1393.5  13.1  +1.0%   2.5   0.9 100.0 120.7s
RC1 DDQN-ALNS       12.7 0.3    +1.1   1387.3  18.4  +0.5%   2.6   1.3 100.0  79.8s
───────────────────────────────────────────────────────────────────────────────────
RC2 ALNS             3.5 0.1    +0.2   1132.2  23.8  +1.6%   3.5   2.1 100.0  91.3s
RC2 DDQN-ALNS        3.6 0.1    +0.4   1121.0  32.3  +0.7%   3.5   3.0 100.0  64.4s
RC2 DDQN-ALNS★       3.6 0.1    +0.4   1121.4  31.9  +0.7%   3.5   3.0 100.0  64.2s
───────────────────────────────────────────────────────────────────────────────────
CV = std/mean×100%. Negative Gap%: solution beats BKS distance.

STATISTICAL TESTS

── Wilcoxon signed-rank (two-sided) ──
Comparison       

In [17]:
# ── Cell 19: Per-Instance Tables (Appendix) ───────────────────────────────────
if not df_all.empty:
    for ds in ['RC1', 'RC2']:
        sub   = df_all[df_all['Dataset'] == ds]
        pivot = sub.pivot_table(
            index='Instance', columns='Algorithm',
            values=['NV_mean', 'TD_mean', 'Gap%', 'NV_cv'],
            aggfunc='mean',
        ).round(2)
        print(f'\n── {ds} per-instance detail ──')
        print(pivot.to_string())


── RC1 per-instance detail ──
           Gap%           NV_cv           NV_mean            TD_mean          
Algorithm  ALNS DDQN-ALNS  ALNS DDQN-ALNS    ALNS DDQN-ALNS     ALNS DDQN-ALNS
Instance                                                                      
RC101     -0.08     -0.89  2.63      3.18    15.2      15.4  1695.62   1681.85
RC102     -2.69     -2.83  3.60      3.60    13.6      13.6  1512.87   1510.68
RC103      4.98      4.44  4.22      3.39    11.6      11.8  1324.48   1317.66
RC104      3.55      3.01  0.00      3.92    10.0      10.2  1175.76   1169.66
RC105     -2.22     -2.46  2.82      3.40    14.2      14.4  1593.32   1589.28
RC106     -1.15     -1.16  3.12      3.12    12.8      12.8  1408.36   1408.14
RC107      4.01      3.20  0.00      0.00    12.0      12.0  1279.80   1269.83
RC108      1.57      1.04  3.70      0.00    10.8      11.0  1157.74   1151.64

── RC2 per-instance detail ──
            Gap%                       NV_cv                      NV_

In [18]:
# ── Cell 20: Route Visualisation ──────────────────────────────────────────────
inst = RC1[0]
for algo, SolverCls in [('ALNS', ALNSSolver), ('DDQN-ALNS', PlateauHybridSolver)]:
    s = SolverCls(inst, CFG)
    plan, _ = s.solve(seed=CFG.seed)
    plot_routes(plan)

✅ Route plot → /kaggle/working/routes_ALNS_RC101.png
✅ Route plot → /kaggle/working/routes_DDQN-ALNS_RC101.png


In [19]:
# ── Cell 21: NEXUS Demo Export ────────────────────────────────────────────────
MAP_INSTANCE = RC1[0]
print(f"Exporting NEXUS demo for {MAP_INSTANCE.name}...")
t0 = time.time()
 
 
def _solve_export(inst: Inst, SolverCls, label: str) -> Dict:
    s         = SolverCls(inst, CFG)
    plan, hist = s.solve(seed=CFG.seed)
    routes_out = []
    for ri, route in enumerate(plan.routes):
        if not route:
            continue
        d = float(inst.dist[0, route[0]])
        for k in range(len(route) - 1):
            d += float(inst.dist[route[k], route[k + 1]])
        d += float(inst.dist[route[-1], 0])
        routes_out.append({
            "id":    ri + 1,
            "nodes": [int(n) for n in route],
            "dist":  round(d, 2),
        })
    bks_td = BKS[inst.name]["td"]
    total  = sum(r["dist"] for r in routes_out)
    gap    = round((total - bks_td) / bks_td * 100, 2)
    print(f"  {label:12s}: nv={len(routes_out)}, td={total:.1f}, gap={gap:+.1f}%")
    return {
        "algo": label, "nv": len(routes_out),
        "td": round(total, 2), "gap_pct": gap,
        "bks_nv": BKS[inst.name]["nv"], "bks_td": bks_td,
        "routes":  routes_out,
        "history": [round(float(c), 2) for c in hist] if hist else [],
    }
 
 
alns_exp = _solve_export(MAP_INSTANCE, ALNSSolver, "ALNS")
rla_exp  = _solve_export(MAP_INSTANCE, PlateauHybridSolver, "DDQN-ALNS")
 
# Real op_counts from last solve
_s = PlateauHybridSolver(MAP_INSTANCE, CFG)
_s.solve(seed=CFG.seed)
op_matrix = [
    [_s.op_counts.get((di, ri), 0) for ri in range(N_R)]
    for di in range(N_D)
]
 
inst = MAP_INSTANCE
nodes_exp = [
    {"id": int(i), "x": float(inst.coords[i, 0]), "y": float(inst.coords[i, 1]),
     "demand": float(inst.demands[i]), "ready": float(inst.ready_times[i]),
     "due": float(inst.due_times[i]),  "svc":   float(inst.service_times[i])}
    for i in range(inst.n + 1)
]
 
summary_exp = [
    {"instance": str(r["Instance"]), "algo": str(r["Algorithm"]),
     "nv": float(r["NV_mean"]),     "td":   float(r["TD_mean"]),
     "gap_pct": float(r["Gap%"]) if pd.notna(r.get("Gap%")) else 0.0,
     "cv_nv":   float(r["NV_cv"]), "cv_td": float(r["TD_cv"]),
     "time_s":  float(r["Time_s"])}
    for _, r in df.iterrows()
]
 
# df_tr may not exist if cell 17 was skipped
transfer_exp = []
if 'df_tr' in dir() and not df_tr.empty:
    transfer_exp = [
        {"instance": str(r["Instance"]), "algo": str(r["Algorithm"]),
         "nv":  float(r["NV_mean"]),    "td":   float(r["TD_mean"]),
         "gap_pct": float(r["Gap%"]) if pd.notna(r.get("Gap%")) else 0.0}
        for _, r in df_tr.iterrows()
    ]
 
OUT = {
    "meta": {
        "instance":    MAP_INSTANCE.name,
        "n_customers": int(MAP_INSTANCE.n),
        "capacity":    float(MAP_INSTANCE.capacity),
        "horizon":     float(MAP_INSTANCE.horizon),
        "dataset":     "Solomon RC1+RC2",
        "version":     "v9.5",
        "algo_desc": {
            "ALNS":       "Adaptive Large Neighbourhood Search (baseline)",
            "DDQN-ALNS":  "Dueling DDQN selects search MODE inside ALNS (proposed)",
            "DDQN-ALNS★": "Zero-shot transfer: trained on RC1, applied to RC2",
            "DQN":        "Constructive RL without metaheuristic (ablation)",
        },
    },
    "nodes":       nodes_exp,
    "alns":        alns_exp,
    "rl_alns":     rla_exp,
    "op_matrix":   op_matrix,
    "destroy_ops": ["Random", "Worst", "Shaw", "Route", "TW-Urgent"],
    "repair_ops":  ["Greedy", "Regret-2", "Regret-3", "TW-Greedy"],
    "summary":     summary_exp,
    "transfer":    transfer_exp,
}
 
out_json = os.path.join(OUTPUT_DIR, "nexus_demo.json")
with open(out_json, "w") as f:
    json.dump(OUT, f, separators=(",", ":"))
 
size_kb = os.path.getsize(out_json) / 1024
print(f"\n✅ nexus_demo.json → {out_json}  ({size_kb:.1f} KB)")
print(f"   Total export time: {time.time() - t0:.1f}s")
print("   Drop nexus_demo.json into nexus_v2.html to view")
 

Exporting NEXUS demo for RC101...
  ALNS        : nv=16, td=1686.8, gap=-0.6%
  DDQN-ALNS   : nv=15, td=1674.9, gap=-1.3%

✅ nexus_demo.json → /kaggle/working/nexus_demo.json  (44.4 KB)
   Total export time: 303.8s
   Drop nexus_demo.json into nexus_v2.html to view
